# Proyecto EDA: Análisis de Campañas de Marketing Bancario
## Fase 1: Carga, Inspección Inicial y Consolidación de Datos

En este cuaderno realizaremos el primer paso del análisis exploratorio de datos (EDA):
1. **Importación de librerías esenciales** (pandas, 
umpy, matplotlib, seaborn).
2. **Carga e inspección de ank-additional.csv** (campañas de marketing y contexto macroeconómico).
3. **Carga e integración de customer-details.xlsx** (datos demográficos en 3 hojas anuales: 2012, 2013 y 2014).
4. **Cruce (Merge / Join) de ambos conjuntos de datos** mediante la clave de cliente (id_ / ID).
5. **Diagnóstico inicial de calidad de datos** (dimensiones, tipos de variables y valores nulos).

---
> 💡 **Nota para desarrolladores .NET / C#**:
> Piensa en un DataFrame de Pandas como una combinación entre un DataTable en memoria y un IQueryable<T>. Métodos como .head(), .info() y .describe() equivalen a inspeccionar el esquema, los tipos y los primeros registros de tus entidades.


In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configuración de visualización para ver todas las columnas en pantalla
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: '%.3f' % x)

print('Librerías importadas correctamente.')
print('Versión de Pandas:', pd.__version__)


### 1. Carga de los datos de campaña (ank-additional.csv)
Usamos pd.read_csv(). Observa que el CSV incluye una columna inicial Unnamed: 0, típica cuando un DataFrame se guardó anteriormente sin especificar index=False. La descartaremos o usaremos como índice.


In [ ]:
csv_path = '../data/raw/bank-additional.csv'
df_campaign = pd.read_csv(csv_path)

print(f'Dimensiones de df_campaign (filas, columnas): {df_campaign.shape}')
df_campaign.head()


### 2. Inspección del esquema y tipos de datos
En C# tienes tipos fuertemente tipados (int, decimal, DateTime, etc.). En Pandas, los tipos se denominan dtypes:
- int64 / loat64: números enteros y flotantes.
- object: cadenas de texto (string) o tipos mixtos.
- datetime64: fechas y horas.

Con .info() vemos rápidamente el número de valores no nulos por columna y el consumo de memoria.


In [ ]:
df_campaign.info()


### 3. Carga e integración del archivo Excel (customer-details.xlsx)
El archivo contiene información demográfica de los clientes repartida en 3 hojas: '2012', '2013' y '2014'.
Para procesarlo eficientemente:
1. Usamos pd.ExcelFile para inspeccionar las hojas sin cargar todo el archivo repetidamente.
2. Leemos cada hoja con pd.read_excel() y añadimos una columna de procedencia (signup_cohort).
3. Concatenamos las 3 hojas verticalmente usando pd.concat() (equivalente a UNION ALL en SQL o .Concat() en LINQ).


In [ ]:
excel_path = '../data/raw/customer-details.xlsx'
xl = pd.ExcelFile(excel_path)
print('Hojas encontradas en el Excel:', xl.sheet_names)

# Leemos cada hoja y las combinamos en una sola lista de DataFrames
sheets_data = []
for sheet_name in xl.sheet_names:
    temp_df = pd.read_excel(xl, sheet_name=sheet_name)
    temp_df['signup_cohort'] = sheet_name  # Registramos el año de la pestaña
    sheets_data.append(temp_df)

# pd.concat une los DataFrames uno debajo de otro (verticalmente)
df_customers = pd.concat(sheets_data, ignore_index=True)

print(f'Dimensiones de df_customers (filas, columnas): {df_customers.shape}')
df_customers.head()


### 4. Verificación de claves primarias y cruce (Merge / Join)
Antes de hacer un cruce, siempre comprobamos la cardinalidad y unicidad de las claves:
- En df_campaign: la clave es id_.
- En df_customers: la clave es ID.

¿Tienen duplicados? ¿Coinciden los registros?


In [ ]:
# Verificamos si los IDs son únicos (equivalente a Primary Key)
print('IDs únicos en campaña:', df_campaign['id_'].nunique(), 'de', len(df_campaign))
print('IDs únicos en clientes:', df_customers['ID'].nunique(), 'de', len(df_customers))

# Analizamos cuántos clientes del Excel no fueron contactados en la campaña
campaign_ids = set(df_campaign['id_'])
customer_ids = set(df_customers['ID'])

print(f'Clientes presentes en ambas bases: {len(campaign_ids.intersection(customer_ids))}')
print(f'Clientes en Excel sin llamadas registradas: {len(customer_ids - campaign_ids)}')


### 5. Fusión de ambos conjuntos (Merge)
En Pandas:
- pd.merge(..., how='inner'): Equivale a INNER JOIN en SQL o .Join(...) en LINQ. Mantiene solo los clientes con registros en ambas tablas.
- pd.merge(..., how='left'): Equivale a LEFT JOIN.

Dado que nuestro objetivo es analizar la efectividad de las campañas y si contrataron (y), nos interesa la población contactada (los 43.000 registros).


In [ ]:
# Realizamos el merge por las columnas id_ e ID
df_merged = pd.merge(
    df_campaign,
    df_customers,
    left_on='id_',
    right_on='ID',
    how='inner'
)

# Eliminamos columnas redundantes (Unnamed: 0 de ambos archivos y el ID repetido)
cols_to_drop = [c for c in df_merged.columns if 'Unnamed' in c] + ['ID']
df_merged = df_merged.drop(columns=cols_to_drop)

print(f'Dimensiones tras el merge: {df_merged.shape}')
df_merged.head(3)


### 6. Diagnóstico inicial de calidad de datos (Missing values y duplicados)
Veamos qué columnas tienen valores nulos (NaN) y en qué proporción.
En LINQ harías db.Where(x => x.Col == null).Count(). En Pandas es df.isna().sum().


In [ ]:
# Calculamos número y porcentaje de valores nulos por columna
null_summary = pd.DataFrame({
    'Valores_Nulos': df_merged.isna().sum(),
    'Porcentaje_%': (df_merged.isna().sum() / len(df_merged)) * 100
})

# Filtramos solo las columnas que tienen algún valor nulo y ordenamos descendente
null_summary = null_summary[null_summary['Valores_Nulos'] > 0].sort_values(by='Valores_Nulos', ascending=False)
null_summary


### 7. Distribución inicial de la variable objetivo (y)
La variable y indica si el cliente contrató el depósito a plazo fijo (yes / 
o).
Con .value_counts() (opcionalmente 
ormalize=True para ver el porcentaje) vemos si estamos ante un dataset desbalanceado, algo muy común en problemas de marketing y riesgo bancario.


In [ ]:
print('Conteo absoluto:')
print(df_merged['y'].value_counts(dropna=False))

print('\nPorcentaje (%):')
print(df_merged['y'].value_counts(normalize=True, dropna=False) * 100)
